# Evaluation Results

This notebook loads the completed automatic evaluation metrics and creates three benchmark-specific presentation figures using only the four headline metrics reported in the results table: POPE Adv. F1, Object HalBench CHAIRS, AMBER CHAIR, and AMBER Hal.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

EVAL_RUN = REPO_ROOT / 'output' / 'eval' / 'stage5_current_parts_eval_full_20260427_012411_retry2'
FIGURE_DIR = REPO_ROOT / 'reports' / 'result_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    ('llava15_base_7b', 'Base'),
    ('direct_paper_hsa_b32_e1', 'DPO-HSA'),
    ('direct_normal_dpo_b32_e1', 'DPO'),
    ('api_verified_margin_hsa_b32_e1', 'HSA-DPO-M'),
]

MODEL_COLORS = {
    'Base': '#2F80ED',
    'DPO-HSA': '#F2994A',
    'DPO': '#27AE60',
    'HSA-DPO-M': '#9B51E0',
}

def read_metric(model_id, benchmark, metric):
    path = EVAL_RUN / 'models' / model_id / 'metrics' / f'{benchmark}.json'
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)
    return payload.get('metrics', {}).get(metric)

rows = []
for model_id, label in MODELS:
    rows.append({
        'model_id': model_id,
        'model': label,
        'POPE Adv. F1\n(higher better)': read_metric(model_id, 'pope_adv', 'f1'),
        'Object HalBench CHAIRS\n(lower better)': read_metric(model_id, 'object_halbench', 'chairs'),
        'AMBER CHAIR\n(lower better)': read_metric(model_id, 'amber', 'amber_chair'),
        'AMBER Hal\n(lower better)': read_metric(model_id, 'amber', 'amber_hal'),
    })

results_df = pd.DataFrame(rows)
display(results_df.drop(columns=['model_id']).round(2))
print(f'Figures will be saved to: {FIGURE_DIR.relative_to(REPO_ROOT)}')

## Main Results

- **POPE Adv. F1:** the base model remains slightly best.
- **Object HalBench CHAIRS:** all trained models reduce hallucination compared with the base model; lower is better.
- **AMBER CHAIR / Hal:** trained models reduce hallucination compared with the base model; lower is better.

## Presentation Figures

The following cells save exactly three benchmark figures. Each figure uses only the headline table numbers and compares the four variants: `Base`, `DPO-HSA`, `DPO`, and `HSA-DPO-M`. No mean line or mean annotation is included.

In [ ]:
def plot_single_metric(metric, title, ylabel, filename, color):
    plot_df = results_df[['model', metric]].copy()

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        plot_df['model'],
        plot_df[metric],
        color=[MODEL_COLORS[m] for m in plot_df['model']],
        edgecolor='white',
        linewidth=0.8,
        alpha=0.92,
    )

    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Model variant')
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(0, max(plot_df[metric]) * 1.18)
    plt.tight_layout()

    out_path = FIGURE_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    return out_path

def plot_amber_metrics(filename):
    metrics = [
        ('AMBER CHAIR\n(lower better)', 'AMBER CHAIR'),
        ('AMBER Hal\n(lower better)', 'AMBER Hal'),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=False)

    for ax, (metric, title) in zip(axes, metrics):
        plot_df = results_df[['model', metric]].copy()
        bars = ax.bar(
            plot_df['model'],
            plot_df[metric],
            color=[MODEL_COLORS[m] for m in plot_df['model']],
            edgecolor='white',
            linewidth=0.8,
            alpha=0.92,
        )
        ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Model variant')
        ax.set_ylabel('Score - lower is better')
        ax.grid(axis='y', alpha=0.25)
        ax.set_ylim(0, max(plot_df[metric]) * 1.18)

    fig.suptitle('AMBER Evaluation', fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    out_path = FIGURE_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    return out_path


In [ ]:
saved_figures = [
    plot_single_metric(
        'POPE Adv. F1\n(higher better)',
        'POPE Adversarial Evaluation',
        'F1 score (%) - higher is better',
        'pope_benchmark_histogram.png',
        '#2F80ED',
    ),
    plot_single_metric(
        'Object HalBench CHAIRS\n(lower better)',
        'Object HalBench Evaluation',
        'CHAIRS - lower is better',
        'object_halbench_histogram.png',
        '#F2994A',
    ),
    plot_amber_metrics('amber_benchmark_histogram.png'),
]

display(pd.DataFrame({
    'figure': [p.name for p in saved_figures],
    'path': [str(p.relative_to(REPO_ROOT)) for p in saved_figures],
}))
